In [1]:
import json
import numpy as np
import pandas as pd

with open("processed_assessments_clean.json", "r", encoding="utf-8") as f:
    assessments = json.load(f)

df_assess = pd.DataFrame(assessments)

print("Total assessments:", len(df_assess))
df_assess.head()

Total assessments: 377


,name,url,assessed_skills_norm,cognitive_dimensions_norm,target_roles_norm,industry_tags,final_duration,experience_numeric,embedding
0,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,"[portability, application development, perform...",[],"[programmers, application developers]",[Information Technology],30.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.019559728, -0.0246569775, 0.0208423324, -0..."
1,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,[],[],[],[],NaN,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[0.0381163061, 0.0718954131, 0.0741915777, 0.0..."
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,"[programming knowledge, validation, routing, s...",[],"[full stack developer, software engineer, tech...",[Information Technology],17.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0091075459, -0.0168671068, 0.0248628519, -..."
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,"[wpf data bindings, mvvm pattern, events, data...",[],"[full stack developer, net developer, software...",[Information Technology],5.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0426559374, 0.017746916, 0.0271624029, -0...."
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,"[behaviors, messages, clients, services and co...",[],"[full stack developer, net developer - wcf, so...",[Information Technology],11.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0292214733, -0.0284445733, 0.0452607013, -..."


In [2]:
embedding_matrix = np.vstack(df_assess["embedding"].values)

print("Embedding matrix shape:", embedding_matrix.shape)

Embedding matrix shape: (377, 1536)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

def get_top_k_similar(query_embedding, k=20):
    
    sims = cosine_similarity([query_embedding], embedding_matrix)[0]
    
    top_indices = np.argsort(sims)[::-1][:k]
    
    results = df_assess.iloc[top_indices].copy()
    results["similarity"] = sims[top_indices]
    
    return results

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="")

In [7]:
from openai import OpenAI

#client = OpenAI()

def embed_query(query):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    return response.data[0].embedding

In [8]:
def extract_query_skills(query):

    prompt = f"""
    Extract the important hiring skills from this query.

    Return JSON:
    {{
      "skills": []
    }}

    QUERY:
    {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You extract hiring skills."},
            {"role": "user", "content": prompt}
        ]
    )

    import re, json

    content = response.choices[0].message.content
    
    match = re.search(r"\{.*\}", content, re.DOTALL)
    
    if match:
        return json.loads(match.group())["skills"]
    
    return []

In [9]:
def compute_skill_overlap(query_skills, assessment_skills):

    if not query_skills:
        return 0
    
    q = set([s.lower() for s in query_skills])
    a = set([s.lower() for s in assessment_skills])
    
    return len(q.intersection(a)) / len(q)

In [10]:
def rerank_results(query_skills, candidates):

    scores = []

    for _, row in candidates.iterrows():

        skill_score = compute_skill_overlap(
            query_skills,
            row["assessed_skills_norm"]
        )

        final_score = (
            0.7 * row["similarity"]
            + 0.3 * skill_score
        )

        scores.append(final_score)

    candidates["final_score"] = scores

    return candidates.sort_values(
        "final_score",
        ascending=False
    )

In [11]:
def recommend_assessments(query, top_n=10):

    # embed query
    q_emb = embed_query(query)

    # semantic retrieval
    candidates = get_top_k_similar(q_emb, k=20)

    # extract skills
    query_skills = extract_query_skills(query)

    # rerank
    ranked = rerank_results(query_skills, candidates)

    return ranked.head(top_n)

In [21]:
query = """
ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 mins long

"""

recommendations = recommend_assessments(query)

recommendations[["name", "url", "final_score"]]

,name,url,final_score
339,Typing (New),https://www.shl.com/products/product-catalog/v...,0.304462
62,Contact Center Call Simulation (New),https://www.shl.com/products/product-catalog/v...,0.298821
293,SAP Basis (New),https://www.shl.com/products/product-catalog/v...,0.294570
10,Accounts Receivable Simulation (New),https://www.shl.com/products/product-catalog/v...,0.288257
113,Financial and Banking Services (New),https://www.shl.com/products/product-catalog/v...,0.281429
208,MS Office Basic Computer Literacy (Sim) (New),https://www.shl.com/products/product-catalog/v...,0.278921
294,SAP Business Objects WebI (New),https://www.shl.com/products/product-catalog/v...,0.276943
8,Accounts Payable Simulation (New),https://www.shl.com/products/product-catalog/v...,0.273215
367,Workplace Administration Skills (New),https://www.shl.com/products/product-catalog/v...,0.273071
308,Siebel Development (New),https://www.shl.com/products/product-catalog/v...,0.272641


In [14]:
import pandas as pd

file_path = "Gen_AI Dataset.xlsx"

df_eval = pd.read_excel(
    file_path,
    sheet_name="Train-Set"
)

df_eval.head()

,Query,Assessment_url
0,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
1,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
2,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
3,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
4,I am hiring for Java developers who can also c...,https://www.shl.com/products/product-catalog/v...


In [15]:
df_eval = df_eval.dropna(subset=["Query", "Assessment_url"])

df_eval["Query"] = df_eval["Query"].str.strip()
df_eval["Assessment_url"] = df_eval["Assessment_url"].str.strip()

print("Total rows:", len(df_eval))

Total rows: 65


In [16]:
ground_truth = (
    df_eval
    .groupby("Query")["Assessment_url"]
    .apply(list)
    .to_dict()
)

queries = list(ground_truth.keys())

print("Unique queries:", len(queries))

Unique queries: 10


In [17]:
def recall_at_10_for_query(query):

    results = recommend_assessments(query, top_n=10)

    predicted_urls = set(results["url"].tolist())

    true_urls = set(ground_truth[query])

    hits = len(predicted_urls.intersection(true_urls))

    recall = hits / len(true_urls)

    return recall

In [18]:
recall_scores = []

for query in queries:
    
    r = recall_at_10_for_query(query)
    
    recall_scores.append({
        "query": query,
        "recall@10": r
    })

df_results = pd.DataFrame(recall_scores)

df_results.head()

,query,recall@10
0,Based on the JD below recommend me assessment ...,0.200000
1,"Content Writer required, expert in English and...",0.000000
2,Find me 1 hour long assesment for the below jo...,0.000000
3,I am hiring for Java developers who can also c...,0.000000
4,I am looking for a COO for my company in China...,0.166667


In [19]:
mean_recall_10 = df_results["recall@10"].mean()

print("Mean Recall@10:", round(mean_recall_10, 4))

Mean Recall@10: 0.0367


In [20]:
df_results.to_csv("recall10_results.csv", index=False)

xxxxxxxxxxxx----------xxxxxxxxxx

In [22]:
import re
import json

def parse_query(query):

    prompt = f"""
Extract structured hiring requirements.

Return JSON:

{{
 "role": "",
 "skills": [],
 "max_duration": null
}}

Query:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system","content":"You extract hiring information."},
            {"role":"user","content":prompt}
        ]
    )

    text = response.choices[0].message.content

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if match:
        return json.loads(match.group())

    return {"role":None,"skills":[],"max_duration":None}

In [23]:
def expand_role_skills(role):

    if role is None or role == "":
        return []

    prompt = f"""
List common skills required for this job role.

Return JSON:
{{
 "skills": []
}}

Role:
{role}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system","content":"You know job skill requirements."},
            {"role":"user","content":prompt}
        ]
    )

    text = response.choices[0].message.content

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if match:
        return json.loads(match.group())["skills"]

    return []

In [24]:
def role_match_score(query_role, target_roles):

    if not query_role or not target_roles:
        return 0

    query_role = query_role.lower()

    for r in target_roles:
        if query_role in r.lower() or r.lower() in query_role:
            return 1

    return 0

In [25]:
def greedy_skill_cover(candidates, required_skills, top_n=10):

    selected = []
    covered = set()

    skills = set([s.lower() for s in required_skills])

    candidates = candidates.copy()

    while len(selected) < top_n and len(covered) < len(skills):

        best_idx = None
        best_gain = 0

        for idx,row in candidates.iterrows():

            assessment_skills = set([s.lower() for s in row["assessed_skills_norm"]])

            gain = len((assessment_skills & skills) - covered)

            if gain > best_gain:
                best_gain = gain
                best_idx = idx

        if best_idx is None:
            break

        selected.append(candidates.loc[best_idx])

        covered |= set(candidates.loc[best_idx]["assessed_skills_norm"])

        candidates = candidates.drop(best_idx)

    remaining = candidates.head(top_n-len(selected))

    selected_df = pd.DataFrame(selected)

    return pd.concat([selected_df, remaining]).head(top_n)

In [26]:
def improved_rerank(query_role, query_skills, candidates):

    scores = []

    for _,row in candidates.iterrows():

        skill_score = compute_skill_overlap(
            query_skills,
            row["assessed_skills_norm"]
        )

        role_score = role_match_score(
            query_role,
            row["target_roles_norm"]
        )

        score = (
            0.5 * row["similarity"] +
            0.3 * skill_score +
            0.2 * role_score
        )

        scores.append(score)

    candidates["final_score"] = scores

    return candidates.sort_values("final_score", ascending=False)

In [27]:
def recommend_assessments(query, top_n=10):

    parsed = parse_query(query)

    role = parsed["role"]
    skills = parsed["skills"]

    role_skills = expand_role_skills(role)

    all_skills = list(set(skills + role_skills))

    query_emb = embed_query(query)

    candidates = get_top_k_similar(query_emb, k=40)

    ranked = improved_rerank(role, all_skills, candidates)

    final_results = greedy_skill_cover(
        ranked,
        all_skills,
        top_n
    )

    return final_results

In [28]:
import pandas as pd

file_path = "Gen_AI Dataset.xlsx"

df_eval = pd.read_excel(
    file_path,
    sheet_name="Train-Set"
)

df_eval.head()

,Query,Assessment_url
0,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
1,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
2,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
3,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
4,I am hiring for Java developers who can also c...,https://www.shl.com/products/product-catalog/v...


In [29]:
def normalize_url(url):

    if pd.isna(url):
        return None

    url = url.lower().strip()

    url = url.replace(
        "/solutions/products/product-catalog/view/",
        "/products/product-catalog/view/"
    )

    return url.rstrip("/")

In [30]:
df_eval["Assessment_url"] = df_eval["Assessment_url"].apply(normalize_url)
df_assess["url"] = df_assess["url"].apply(normalize_url)

In [31]:
ground_truth = (
    df_eval
    .groupby("Query")["Assessment_url"]
    .apply(list)
    .to_dict()
)

queries = list(ground_truth.keys())

print("Total unique queries:", len(queries))

Total unique queries: 10


In [32]:
def recall_at_10_for_query(query):

    results = recommend_assessments(query, top_n=10)

    predicted_urls = set(results["url"].tolist())

    true_urls = set(ground_truth[query])

    hits = len(predicted_urls.intersection(true_urls))

    recall = hits / len(true_urls)

    return recall

In [33]:
recall_scores = []

for query in queries:

    r = recall_at_10_for_query(query)

    recall_scores.append({
        "query": query,
        "recall@10": r
    })

df_recall = pd.DataFrame(recall_scores)

df_recall.head()

,query,recall@10
0,Based on the JD below recommend me assessment ...,0.200000
1,"Content Writer required, expert in English and...",0.600000
2,Find me 1 hour long assesment for the below jo...,0.222222
3,I am hiring for Java developers who can also c...,0.400000
4,I am looking for a COO for my company in China...,0.166667


In [34]:
mean_recall_10 = df_recall["recall@10"].mean()

print("Mean Recall@10:", round(mean_recall_10, 4))

Mean Recall@10: 0.23


In [36]:
df_recall.to_csv("recall10_results1.csv", index=False)